In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028866BFBE00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028866DA0C20>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    name:str=Field(description="the name of the movie")
    year:int=Field(description="Movie releases in a year")
    director:str=Field(description="Movie directed by the director")
    rating:float=Field(description="Rating of the movie out of 5")

In [ ]:
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("give the details about movie cocktail")
response

Movie(name='Cocktail', year=1988, director='Roger Donaldson', rating=3.1)

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A Movie with the details."""
    name:str=Field(..., description="the title of the movie")
    actors:str=Field(..., description="the lead starcast of the movie")
    director:str=Field(..., description="the director of the movie")
    rating:float=Field(..., description="Rating of the movie out of 5.0")
    year:int=Field(..., description="The release year of the movie.")

mws = model.with_structured_output(Movie)
mws

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028866BFBE00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028866DA0C20>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': 'A Movie with the details.', 'parameters': {'properties': {'name': {'description': 'the title of the movie', 'type': 'string'}, 'actors': {'description': 'the lead starcast of the movie', 'type': 'string'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'Rating of the movie out of 5.0', 'type': 'number'}, 'year'

In [24]:
response = mws.invoke("give the details about bollywood movie sholay")
response

Movie(name='Sholay', actors='Amitabh Bachchan, Dharmendra, Sanjeev Kumar, Jaya Bhaduri, Hema Malini', director='Ramesh Sippy', rating=4.8, year=1975)

In [28]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
class Movie(BaseModel):
    title:str
    direcotr:str
    cast:list[Actor]
    budget: float | None = Field(None, description="the budget of the movie in INR in CR")
    rating: int

mws = model.with_structured_output(Movie)
ans = mws.invoke("give the details about bollywood movie bahubali2")
ans

Movie(title='Bahubali 2', direcotr='S.S. Rajamouli', cast=[Actor(name='Prabhas', role='Bahubali'), Actor(name='Tamannaah Bhatia', role='Devasena'), Actor(name='Rana Daggubati', role='Bhallala Deva')], budget=350.0, rating=9)

In [32]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    """A movie with details"""
    title:Annotated[str, ..., "The name of the movie"]
    cast:Annotated[str, ..., "The starcast of the movie"]
    rating: Annotated[int, ..., "rating of the movie out of 5"]
    budget: Annotated[float, ..., "The budget of the entire movie in CR"]

mws = model.with_structured_output(Movie)
ans = mws.invoke("give the details about bollywood movie 3 Idiots")
ans

{'budget': 35,
 'cast': 'Aamir Khan, R. Madhavan, Sharman Joshi, Boman Irani',
 'rating': 5,
 'title': '3 Idiots'}

In [6]:
from typing import TypedDict, Annotated
from pydantic import  Field
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

class Actor(TypedDict):
    Actor: str
    role: str
class Movie(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    budget: float
    rating: int=Field(description="rating of the movie out of 5")

mws = model.with_structured_output(Movie)
ans = mws.invoke("Give me the info about movie 3 idiots")
ans

{'budget': 15000000,
 'cast': [{'Actor': 'Aamir Khan', 'role': 'Rancho'},
  {'Actor': 'R. Madhavan', 'role': 'Farhan Qureshi'},
  {'Actor': 'Sharman Joshi', 'role': 'Raju Rastogi'}],
 'rating': 8,
 'title': '3 Idiots',
 'year': '2009'}

In [7]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [13]:
import os
from langchain.agents import create_agent
from pydantic import BaseModel

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str=Field(description="name of the person")
    email:str=Field(description="email address of the person")
    phone:int=Field(description="phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format= ContactInfo
)
ans = agent.invoke({
    "messages": [{'role': 'user', 'content': 'extract the information from this: brijesh kapadiya, brijeshkapadiya@gmail.com, (+91) 84605-56315'}] 
})
ans['structured_response']

ContactInfo(name='Brijesh Kapadiya', email='brijeshkapadiya@gmail.com', phone=918460556315)

In [14]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information of a person"""
    name:str=Field(description="name of the person")
    email:str=Field(description="email address of the person")
    phone:int=Field(description="phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format= ContactInfo
)
ans = agent.invoke({
    "messages": [{'role': 'user', 'content': 'extract the information from this: brijesh kapadiya, brijeshkapadiya@gmail.com, (+91) 84605-56315'}] 
})
ans['structured_response']

{'name': 'Brijesh Kapadiya',
 'email': 'brijeshkapadiya@gmail.com',
 'phone': 918460556315}

In [15]:
from dataclasses import  dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo():
    """Contact information of a person"""
    name:str=Field(description="name of the person")
    email:str=Field(description="email address of the person")
    phone:int=Field(description="phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format= ContactInfo
)
ans = agent.invoke({
    "messages": [{'role': 'user', 'content': 'extract the information from this: brijesh kapadiya, brijeshkapadiya@gmail.com, (+91) 84605-56315'}] 
})
ans['structured_response']

ContactInfo(name='Brijesh Kapadiya', email='brijeshkapadiya@gmail.com', phone=918460556315)